# Prototype notebook

This notebook is the original exploration: it is where the chunking, embedding,
storage and retrieval steps were worked out step by step.

**It is not the application.** The cleaned-up, tested version of this pipeline
lives in [`src/rag/`](../src/rag) and is what the CLI and the Streamlit app use:

```bash
python -m rag ingest      # build the index
python -m rag chat        # ask questions
streamlit run app.py      # web UI
```

The notebook is kept because it documents how the design was reached.
Configuration is read from `.env` (see `.env.example`) - no credentials here.


# Connection à la base de donné


In [ ]:
import sys

sys.path.insert(0, "../src")

from rag.config import settings

# Configuration comes from ../.env - see .env.example. Nothing secret is
# written into this notebook.
conn_info = settings.dsn
print("Connecting to:", settings.pg_db, "on", settings.pg_host)


In [2]:
with psycopg.connect(conn_info) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id SERIAL PRIMARY KEY,
                corpus_text TEXT NOT NULL,
                embedding_float DOUBLE PRECISION[]
            );
        """)
        conn.commit()

# chunking


In [3]:
import os
from pathlib import Path

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

def process_all_documents(data_directory: str):
    """Process all PDF and TXT files in a directory (recursively)."""
    all_documents = []
    data_dir = Path(data_directory)

    # Find all PDF and TXT files recursively
    files = list(data_dir.rglob("*.pdf")) + list(data_dir.rglob("*.txt"))

    print(f"Found {len(files)} files to process")

    for file_path in files:
        print(f"\nProcessing: {file_path.name}")
        try:
            # Choose the right loader depending on extension
            suffix = file_path.suffix.lower()
            if suffix == ".pdf":
                loader = PyPDFLoader(str(file_path))
                file_type = "pdf"
            elif suffix == ".txt":
                # encoding='utf-8' to avoid some common errors
                loader = TextLoader(str(file_path), encoding="utf-8")
                file_type = "txt"
            else:
                # Should not happen because we filtered above, but just in case
                print(f"  ✗ Unsupported file type: {suffix}")
                continue

            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = file_path.name
                doc.metadata["file_type"] = file_type

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages/chunks")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Example: process all PDFs + TXTs in ../data
all_documents = process_all_documents("../data")


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks = split_documents(all_documents)

c:\Users\User\Desktop\Chatbot-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 43 files to process

Processing: Pres_Accueil_UBS.pdf
  ✓ Loaded 9 pages/chunks

Processing: accueil_ubs.pdf
  ✓ Loaded 36 pages/chunks

Processing: 017_00000012.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\017_00000012.txt

Processing: 018_00000013.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\018_00000013.txt

Processing: 019_00000014.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\019_00000014.txt

Processing: 020_00000015.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\020_00000015.txt

Processing: 022_00000017.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\022_00000017.txt

Processing: 023_00000018.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\023_00000018.txt

Processing: 024_00000019.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\024_00000019.txt

Processing: 027_0000001c.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\027_0000001c.txt

Processing: 028_0000001d.txt
  ✗ Error: Error loading ..\data\TRANS_TXT\028_0000001d.txt

Processing: 029_0000001e.txt
  ✗ Error: Error l

# Embedding

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer #to generate embeddings using transformer models from huggingface

from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


# fill the base 

In [7]:
# 1) Récupérer le texte de chaque chunk
chunk_texts = [doc.page_content for doc in chunks]
print(f"Nombre de chunks : {len(chunk_texts)}")

# 2) Générer les embeddings
embeddings = embedding_manager.generate_embeddings(chunk_texts)
print(f"Shape des embeddings : {embeddings.shape}")  # (nb_chunks, dim)

# 3) Insérer dans la base
insert_query = """
    INSERT INTO documents (corpus_text, embedding_float)
    VALUES (%s, %s)
"""

# psycopg sait convertir une liste Python -> DOUBLE PRECISION[]
rows_to_insert = [
    (text, emb.tolist())          # emb est un np.array, on le convertit en list
    for text, emb in zip(chunk_texts, embeddings)
]

with psycopg.connect(conn_info) as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_query, rows_to_insert)
    conn.commit()

print("✅ Tous les chunks et leurs embeddings ont été insérés dans la table documents.")


Nombre de chunks : 116
Generating embeddings for 116 texts...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches: 100%|██████████| 4/4 [00:08<00:00,  2.25s/it]


Generated embeddings with shape: (116, 384)
Shape des embeddings : (116, 384)
✅ Tous les chunks et leurs embeddings ont été insérés dans la table documents.


In [8]:
import psycopg
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Optional

class PostgresRAGRetriever:
    """Handles query-based retrieval from a PostgreSQL table storing embeddings"""
    
    def __init__(
        self, 
        conn_info: str, 
        embedding_manager: EmbeddingManager, 
        table_name: str = "documents"
    ):
        """
        Initialize the retriever
        
        Args:
            conn_info: psycopg connection string
            embedding_manager: Manager for generating query embeddings
            table_name: Name of the table containing corpus_text + embedding_float[]
        """
        self.conn_info = conn_info
        self.embedding_manager = embedding_manager
        self.table_name = table_name

    def _fetch_all_documents(self) -> Dict[str, Any]:
        """Fetch all documents and embeddings from PostgreSQL"""
        with psycopg.connect(self.conn_info) as conn:
            with conn.cursor() as cur:
                cur.execute(f"""
                    SELECT id, corpus_text, embedding_float
                    FROM {self.table_name}
                    WHERE embedding_float IS NOT NULL
                """)
                rows = cur.fetchall()
        
        if not rows:
            print("⚠️ No documents with embeddings found in the database.")
            return {
                "ids": [],
                "texts": [],
                "embeddings": np.array([]),
            }
        
        ids = []
        texts = []
        embeddings = []
        
        for row in rows:
            doc_id, text, emb_list = row
            ids.append(doc_id)
            texts.append(text)
            # emb_list should be a list/tuple of floats → convert to np.array
            embeddings.append(np.array(emb_list, dtype=np.float32))
        
        # Stack into a matrix (n_docs, dim)
        emb_matrix = np.vstack(embeddings)
        
        return {
            "ids": ids,
            "texts": texts,
            "embeddings": emb_matrix,
        }

    def retrieve(
        self, 
        query: str, 
        top_k: int = 5, 
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum cosine similarity score threshold (0–1)
            
        Returns:
            List of dictionaries containing retrieved documents and scores
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # 1) Embedding de la requête
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]  # (dim,)
        query_embedding = query_embedding.reshape(1, -1)  # (1, dim)
        
        # 2) Récupérer les docs depuis PostgreSQL
        data = self._fetch_all_documents()
        ids = data["ids"]
        texts = data["texts"]
        doc_embeddings = data["embeddings"]
        
        if len(ids) == 0:
            return []
        
        # 3) Similarité cosinus entre query et chaque doc
        similarities = cosine_similarity(query_embedding, doc_embeddings)[0]  # shape: (n_docs,)
        
        # 4) Trier par similarité décroissante
        sorted_indices = np.argsort(similarities)[::-1]  # descending
        
        retrieved_docs: List[Dict[str, Any]] = []
        
        for rank_idx, idx in enumerate(sorted_indices[:top_k]):
            score = float(similarities[idx])
            if score < score_threshold:
                continue
            
            retrieved_docs.append({
                "id": ids[idx],
                "content": texts[idx],
                "similarity_score": score,
                "rank": rank_idx + 1
            })
        
        print(f"Retrieved {len(retrieved_docs)} documents (after filtering).")
        return retrieved_docs


In [9]:
rag_retriever = PostgresRAGRetriever(conn_info=conn_info, embedding_manager=embedding_manager)

query = "qu'est-ce que UBS ?"
results = rag_retriever.retrieve(query, top_k=5, score_threshold=0.3)

for r in results:
    print("\n---")
    print(f"Rank: {r['rank']}, Score: {r['similarity_score']:.3f}")
    print(r['content'][:300], "...")


Retrieving documents for query: 'qu'est-ce que UBS ?'
Top K: 5, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.42it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents (after filtering).

---
Rank: 1, Score: 0.468
<19> client 
     c: merci 
<20> hotesse 
     h: # eh Prénom # Prénom2 n'est pas là # Prénom2 n'est pas là # ah d'accord # 
alors e mademoiselle la secrétaire commence à neuf heures moins le quart donc on est 
un petit peu tôt là 
<21> client 
     c: e ...

---
Rank: 2, Score: 0.468
<19> client 
     c: merci 
<20> hotesse 
     h: # eh Prénom # Prénom2 n'est pas là # Prénom2 n'est pas là # ah d'accord # 
alors e mademoiselle la secrétaire commence à neuf heures moins le quart donc on est 
un petit peu tôt là 
<21> client 
     c: e ...

---
Rank: 3, Score: 0.464
c: avant avant onze heures et demie parce que après j'ai je serai pas là 
<28> hotesse 
     h: avant onze heures et demie  
<29> client 
     c: s'il peut 
<30> hotesse 
     h: d'accord 
<31> hotesse+client 
     h: c'est monsieur Nom2 
     c: très bien 
<32> client 
     c: c'est ça 
<33> hotess ...

---
Rank: 4, Score: 0.464
c: avant avant onze heures e

In [10]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../src/.env")

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found in environment variables.")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=2048, # Optionnel, par défaut ChatGroq gère bien cela
)

In [11]:
rag_retriever = PostgresRAGRetriever(conn_info=conn_info, embedding_manager=embedding_manager)
from langchain_core.messages import SystemMessage, HumanMessage
from typing import List, Dict, Any

def rag_simple(
    query: str,
    retriever: PostgresRAGRetriever,
    llm: ChatGroq,
    top_k: int = 5
) -> str:
    """
    Simple RAG: retrieve context from PostgreSQL + generate answer with LLM.
    """
    # 1) Récupérer les chunks pertinents depuis la base
    results: List[Dict[str, Any]] = retriever.retrieve(query, top_k=top_k)
    
    if not results:
        return "No relevant context found in the database to answer the question."
    
    # 2) Construire le contexte à partir du contenu des chunks
    context = "\n\n---\n\n".join(
        [doc["content"] for doc in results]
    )
    
    # 3) Construire le prompt pour le LLM
    system_prompt = (
        "You are a helpful assistant. Use ONLY the provided context to answer "
        "the question as accurately and concisely as possible. "
        "If the context is insufficient, say that you don't know."
    )
    
    user_prompt = f"""
Context:
{context}

Question: {query}

Answer in a clear and concise way:
"""
    
    # 4) Appel au LLM (format messages)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    
    response = llm.invoke(messages)
    return response.content


In [12]:
# On suppose que tout le reste est déjà fait :
# - DB créée
# - chunks insérés avec embeddings
# - embedding_manager initialisé
# - PostgresRAGRetriever initialisé
# - LLM Groq initialisé

question = "what is corpus  Accueil UBS?"
answer = rag_simple(
    query=question,
    retriever=rag_retriever,
    llm=llm,
    top_k=5
)

print("QUESTION:", question)
print("\nRÉPONSE RAG:\n", answer)


Retrieving documents for query: 'what is corpus  Accueil UBS?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.42it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents (after filtering).
QUESTION: what is corpus  Accueil UBS?

RÉPONSE RAG:
 Corpus Accueil_UBS is a pilot corpus of oral human-to-human dialogue, corresponding to a telephone reception task at a university standard. It consists of recorded dialogues between callers and university reception staff, along with orthographic transcriptions of these dialogues.
